# Validating data with Pandera

## Chosen topic

Data validation with a focus on **Pandera**, applied to **email campaign data**.

The goal is to learn how to define and use schemas in Pandera, show which rows pass and which are rejected, as well as why. 

Reliable data is a fundamental prerequisite for analyses and predictive models, which connects directly to my internship where data quality is part of the work.


## Definition

Pandera is an open source project that allows us to perform data validation on dataframe-like objects.
The goal is to **make data processing pipelines more readable and robust**. 

Pandera makes it possible to:  
- Define a schema once and use it to validate different dataframes.  
- Check the types and properties of columns in a pd.DataFrame or values in a pd.Series.
- Parse data to standardize the preprocessing steps needed to produce valid data.
- Integrate with existing data analysis/processing pipelines viafunction decorators.  
- Define dataframe models.   
- Lazily validate dataframes so that all validation rules are executed before raising an error.  
- Integrate with a rich ecosystem of python tools like pydantic and fastapi.


## Starting questions

1. How do I define a schema and what does it check?  
2. What constraints can I put on columns beyond dtype? 
3. What happens when validation fails?
4. How does Pandera fit into a pipeline?
5. What are the limitations with real messy data?

In [14]:
from importlib.metadata import version
import pandas as pd
import pandera.pandas as pa # importing like this follows the current best practices for pandas-specific validation.
from pandera.pandas import (
    Column, 
    DataFrameSchema,
    DataFrameModel, 
    Check
)
from pandera.typing.pandas import Series

print(version('pandera')) # latest stable version 0.33.1 released 1 sept 2026

0.33.1


Sources:
Pandera documentation: https://pandera.readthedocs.io/en/stable/

A schema in Pandera is like a contract that describes the expected structure and properties of the data.  
It is like a blueprint that specifies data types, acceptable value ranges and relationships between columns.
When validating a df against a schema, Pandera checks that every aspect matches the specifications.

**Schemas consist in column definition**, each with data type and optional constraints.

Pandera offers two main approaches: 
- Object-based API
- Class-based API

Validation checks **three key aspects**:
- **Structure** (are all the required columns there?)
- **Types** (is each column the correct datatype?)
- **Values** (do values satisfy all the constraints?)

Source: 
Statology: https://www.statology.org/data-validation-in-python-with-pandera-a-practical-introduction/

In [2]:
# Creating basic email campaign data
data = {
    'subject_line': ['Spring Sale', 'Newsletter #42', 'Spring Sale', 'Product Launch', 'Newsletter #42'],
    'recipients': [1200, 1150, 1200, 1300, 1150],
    'sent_at': ['2026-03-01', '2026-03-15', '2026-03-01', '2026-04-02', '2026-03-15'],
    'opens': [340, 210, 340, 520, 190],
    'cost_per_email': [0.05, 0.08, 0.05, 0.12, 0.08],
}

df = pd.DataFrame(data)
df['sent_at'] = pd.to_datetime(df['sent_at'])

### Basic type validation with DataFrameSchema

In [ ]:
# Creating a schema that validates structure and basic types
schema = DataFrameSchema({
    'subject_line': Column(str),
    'recipients': Column(int),
    'sent_at': Column(pa.DateTime), # to validate dates use pandas dtype string specification Column('datetime64[ns]') or pandera's alias Column(pa.DateTime)
    'opens': Column(int),
    'cost_per_email': Column(float)
})

# Valideting the df
df_validated_schema = schema.validate(df)
# There is no output because the method returns the df silently when it succeeds.  

Source:  
Pandera docs on Pandera's dtypes - https://pandera.readthedocs.io/en/stable/reference/dtypes.html#api-dtypes

### Checks and categorical contraints

They allow to specify properties about a df, columnsm indexes, series. They are applied after dtype validation/coercion.

Multiple checks can be applied to a column.

Also, check objects accept a function as a required argument.

In [4]:
# Built-in checks (operating on pd.series)
enhanced_schema = DataFrameSchema({
    'subject_line': Column(str, Check.isin(['Spring Sale', 'Newsletter #42', 'Product Launch'])),
    'recipients': Column(int, Check.greater_than_or_equal_to(0)),
    'sent_at': Column(pa.DateTime, Check.greater_than_or_equal_to(pd.Timestamp('2026-01-01'))),
    'opens': Column(int, Check.greater_than_or_equal_to(0)),
    'cost_per_email': Column(float, Check.in_range(0, 0.2))
})

validated_df = enhanced_schema.validate(df)

### Coercing a column and allowing missing values

Pandera is primarily a validation library that checks data without changing anything about the dataframe itself.

However, in many cases its useful to parse, i.e. transform the data values to the data contract specified in the schema. Pandera's built in coercion allows to do dtype casting specifically by passing in the `coerce=True` argument to the schema.

Pandera treats all columns as non-nullable. Set `nullable=True` explicitly for columns where missing values are acceptable.

In the checking phase, the null values are ignored. But if it is important that they are included in the check, then specify `Check(..., ignore_na=False)` when defining a check.


Source:  
- https://pandera.readthedocs.io/en/stable/dtypes.html

In [5]:
data_to_coerce = {
    'subject_line': [ 'Spring Sale', 'Newsletter #42', 'Spring Sale', 'Product Launch', None],
    'recipients': ['1200', 1150.0, 1200, 1300, 1150],
    'sent_at': [None, '2026-03-15', '2026-03-01', '2026-04-02', '2026-03-15'],
    'opens': [None, 210, 340.0, 520, 190],
    'cost_per_email': [None, 0.08, 0.05, 0.12, 1],
}

df_to_coerce = pd.DataFrame(data_to_coerce)

schema_with_coerce = DataFrameSchema({
    'subject_line': Column(str, Check.isin(['Spring Sale', 'Newsletter #42', 'Product Launch']), nullable=True),
    'recipients': Column(int, Check.in_range(1, 5000)),
    'sent_at': Column(pa.DateTime, Check.greater_than_or_equal_to(pd.Timestamp('2026-01-01')), nullable=True),
    'opens': Column(pd.Int16Dtype, Check.in_range(0, 5000), nullable=True),
    'cost_per_email': Column(float, Check.in_range(0.0, 1.0), nullable=True)
    },
    coerce=True 
)

coerced_df = schema_with_coerce.validate(df_to_coerce)

coerced_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   subject_line    4 non-null      str           
 1   recipients      5 non-null      int64         
 2   sent_at         4 non-null      datetime64[ns]
 3   opens           4 non-null      Int16         
 4   cost_per_email  4 non-null      float64       
dtypes: Int16(1), datetime64[ns](1), float64(1), int64(1), str(1)
memory usage: 307.0 bytes


### Cross-Column Validation

In [6]:
# Certain column combinations should make sense: opens <= recipients

def check_opens_vs_recipients(df):
    return (df['opens'] <= df['recipients']).all()

full_schema = DataFrameSchema({
    'subject_line': Column(str, Check.isin(['Spring Sale', 'Newsletter #42', 'Product Launch']), nullable=True),
    'recipients': Column(int, Check.in_range(1, 5000)),
    'sent_at': Column(pa.DateTime, Check.greater_than_or_equal_to(pd.Timestamp('2026-01-01')), nullable=True),
    'opens': Column(pd.Int16Dtype, Check.in_range(0, 5000), nullable=True),
    'cost_per_email': Column(float, Check.in_range(0.0, 1.0), nullable=True)
    },
    checks=Check(check_opens_vs_recipients, error='The openings should be equal or less than the recipients.'),
    coerce=True
    )

validated_df = full_schema.validate(df_to_coerce)

validated_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   subject_line    4 non-null      str           
 1   recipients      5 non-null      int64         
 2   sent_at         4 non-null      datetime64[ns]
 3   opens           4 non-null      Int16         
 4   cost_per_email  4 non-null      float64       
dtypes: Int16(1), datetime64[ns](1), float64(1), int64(1), str(1)
memory usage: 307.0 bytes


### Validation failure: lazy=True and Error Reports

By default, when one of the validation requirements is broken, a SchemaError is raised and the validation stopped.

This behavior is fine for simple tasks, but in more complex situations we want to collect all the errors at once. Pandera can provide an error report that includes detailes error information that can be used for logging, data cleaning and user feedback. 

To create an error report with pandas, you must specify `lazy=True` to allow all errors to be aggregated and raised together as a SchemaErrors.

By default, error reports are generated for both schema and data level validation, but more granular control over schema or data only validations is available.

In [7]:
# Creating bad data
bad_data = {
    'subject_line': [ 'Spring Sale', 'Newsletter', 'Spring Sale', 'Product Launch', None],
    'recipients': ['1200', 1150.0, 1200, 1300, 1150],
    'sent_at': [None, '2026-03-15', '2026-03-01', '2026-04-02', '2025-12-30'],
    'opens': [1220, 210, 340.0, 520, 190],
    'cost_per_email': [None, 0.08, 0.05, 0.12, 1],
}

bad_df = pd.DataFrame(bad_data)

In [ ]:
# Running the bad_data df through the full schema to analyze the error

# validate_bad_data = full_schema.validate(bad_df, lazy=True)
# Commented out as it raises SchemaError and interrups the notebook
# uncomment and run to show the message and checks that were broken.

### Handling validation error gracefully

Wrapping the call to validate into a `try/except` block catches pandera's error types.  
Passing `lazy=True` collects every violation across the whole df in one run.  
The `SchemaErrors` that is raised carries `.failure_cases` which is a df listing of which col, indx, value failed and why.  

Then you can decide to log them for a human to review, quarantine/drop just those rows and keep processing the rest, or route them back for cleaning and re-validation, but that decision should be explicit and visible.

In [9]:
try:
    validate_bad_data = full_schema.validate(bad_df, lazy=True)
except pa.errors.SchemaErrors as exc:
    print('Validation failed:')
    display(exc.failure_cases) # List of what failed and why
    display(exc.data) # The df after coearcion

Validation failed:


,schema_context,column,check,check_number,failure_case,index
2,DataFrameSchema,None,The openings should be equal or less than the ...,0,False,None
0,Column,subject_line,"isin(['Spring Sale', 'Newsletter #42', 'Produc...",0,Newsletter,1
1,Column,sent_at,greater_than_or_equal_to(2026-01-01 00:00:00),0,2025-12-30 00:00:00,4


,subject_line,recipients,sent_at,opens,cost_per_email
0,Spring Sale,1200,NaT,1220,NaN
1,Newsletter,1150,2026-03-15,210,0.08
2,Spring Sale,1200,2026-03-01,340,0.05
3,Product Launch,1300,2026-04-02,520,0.12
4,NaN,1150,2025-12-30,190,1.00


### Showing which rows pass and which are rejected

### Raising warnings instead of Error on check failure

The Check and Hypothesis classes and their built-in methods support the keyword argument raise_warning, which is False by default. If set to True, the check will warn with a SchemaWarning instead of raising a SchemaError exception.

Use this feature carefully! If the check is for informational purposes and not critical for data integrity then use raise_warning=True. However, if the assumptions expressed in a Check are necessary conditions to considering your data valid, do not set this option to true.


### Basic type validation with DataFrameModel

`DataFrameModel` is a class-based API where each column becomes a typed class attribute. It is preferred for bigger projects as it scales better. It mirrors the Pydantic pattern in defining schemas as Python classes.

With this short example I wanted to get a general sense of how DataFrameModel works. I will stick with DataFrameSchema as it is the foundation for DataFrameModel and for a project this size I'd rather focusing on the basics of the library.

In [ ]:
class EmailCampaign(pa.DataFrameModel):
    subject_line: Series[str] = pa.Field(isin=['Spring Sale', 'Newsletter #42', 'Product Launch'], coerce=True, nullable=True)
    recipients: Series[int] = pa.Field(ge=0, le=5000, coerce=True)
    sent_at: Series[pa.DateTime] = pa.Field(ge=pd.Timestamp('2026-01-01'), coerce=True, nullable=True)
    opens: Series[int] = pa.Field(ge=0, coerce=True, nullable=True)
    cost_per_email: Series[float] = pa.Field(in_range={'min_value': 0.0, 'max_value': 1}, coerce=True, nullable=True)

    @pa.dataframe_check # Custom check
    def check_opens_vs_recipients(cls, df: pd.DataFrame) -> pd.Series:
        return (df['opens'] <= df['recipients']) 

df_validated_model = EmailCampaign.validate(df_to_coerce, lazy=True)

### Preprocessing with Parsers
https://pandera.readthedocs.io/en/stable/parsers.html

### Dropping invalid rows automatically
https://pandera.readthedocs.io/en/stable/drop_invalid_rows.html

### Pipeline integration
https://pandera.readthedocs.io/en/stable/decorators.html

Notes for later:

- Col requirements: by default the cols in the schema are required. If you want to make a col optional, use required=False. To avoid allowing spelling mistakes to be passed silently, you can set up a warning that informs you about the mising column.
- In case you want to be strict and only accept columns in the schema, strict=True
- Alternatively, if your DataFrame contains columns that are not in the schema, and you would like these to be dropped on validation, you can specify strict='filter'
- You can validate the order of the cols. Many ML libraries will cast a Dataframe to numpy arrays, for which order becomes crucial. Specify ordered=True. Out-of-order columns are reported as SchemaErrors 
- Validating the joint uniqueness of columns
- add_missing_columns=True

Sources:  
- https://pythondatabench.com/article/data-validation-python-pandera-practical-guide
- https://medium.com/towards-artificial-intelligence/your-model-is-fine-your-data-isnt-dataframe-validation-with-pandera-1552c0daeeaf